# 🐾 Model 4: MobileNetV2 Pretrained Transfer Learning
### Member 2: Baseline CNN & Edge Vision Lead (Samiun Mahmud)
**Course**: CSE445 Machine Learning | **Dataset**: 5 Animal Classes (Cat, Cow, Dog, Lamb, Zebra)

---
## 📌 Objectives & Scope (Week 2 & Week 3 Milestones)
1. **Pretrained Backbone**: Leverage ImageNet-trained weights on inverted residual blocks.
2. **Phase 1 (Feature Extraction)**: Freeze the 154-layer backbone; train custom classification head for 10 epochs.
3. **Phase 2 (Fine-Tuning)**: Unfreeze top inverted residual blocks (layers 100+); train with 10x lower learning rate (1e-5).
4. **Edge Vision Benchmark**: Benchmark parameters (~2.26M), inference latency (~8ms), and test accuracy against Model 3 (Custom CNN).

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers, Model
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
from sklearn.metrics import classification_report, confusion_matrix

print(f"TensorFlow Version: {tf.__version__}")

## 🗂 1. Dataset Loading & Standardized Preprocessing

In [ ]:
DATA_DIR = 'dataset'  # Path to dataset folder containing cat, cow, dog, lamb, zebra
IMG_SIZE = (224, 224)
BATCH_SIZE = 32

dataset = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=True,
    seed=42
)

class_names = dataset.class_names
print("Detected Animal Classes:", class_names)

# 70% Train, 15% Val, 15% Test Split
total_batches = len(dataset)
train_size = int(total_batches * 0.70)
val_size = int(total_batches * 0.15)
test_size = total_batches - train_size - val_size

train_ds = dataset.take(train_size).cache().prefetch(tf.data.AUTOTUNE)
val_ds = dataset.skip(train_size).take(val_size).cache().prefetch(tf.data.AUTOTUNE)
test_ds = dataset.skip(train_size + val_size).take(test_size).cache().prefetch(tf.data.AUTOTUNE)

print(f"Train Batches: {train_size}, Val Batches: {val_size}, Test Batches: {test_size}")

## 🧠 2. Model 4 Architecture: MobileNetV2 with ImageNet Weights

In [ ]:
# Base Pretrained Feature Extractor (154 layers)
base_model = MobileNetV2(
    input_shape=(224, 224, 3),
    include_top=False,
    weights='imagenet'
)
base_model.trainable = False  # Freeze backbone for Phase 1

# Custom Classification Head
inputs = layers.Input(shape=(224, 224, 3), name="input_tensor")
x = preprocess_input(inputs)
x = base_model(x, training=False)
x = layers.GlobalAveragePooling2D(name="gap")(x)
x = layers.Dropout(0.3, name="top_dropout")(x)
outputs = layers.Dense(5, activation="softmax", name="animal_predictions")(x)

model = Model(inputs, outputs, name="Animal_Classifier_MobileNetV2")
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)
model.summary()

## 🚀 3. Phase 1 Training: Feature Extraction (Frozen Backbone)

In [ ]:
EPOCHS_PHASE1 = 10
hist_p1 = model.fit(
    train_ds,
    epochs=EPOCHS_PHASE1,
    validation_data=val_ds
)

## ⚙️ 4. Phase 2 Training: Fine-Tuning Top Inverted Residual Blocks

In [ ]:
base_model.trainable = True
# Freeze early low-level feature layers (first 100 layers)
for layer in base_model.layers[:100]:
    layer.trainable = False

# Recompile with 10x-100x lower learning rate
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

EPOCHS_PHASE2 = 10
hist_p2 = model.fit(
    train_ds,
    epochs=EPOCHS_PHASE2,
    validation_data=val_ds
)

## 📊 5. Evaluation, Confusion Matrix & Benchmark vs Model 3

In [ ]:
y_true = []
y_pred = []

for X, y in test_ds:
    preds = model.predict(X, verbose=0)
    y_true.extend(y.numpy())
    y_pred.extend(np.argmax(preds, axis=1))

print("\n=================== CLASSIFICATION REPORT ===================")
print(classification_report(y_true, y_pred, target_names=class_names))
print("\n=================== CONFUSION MATRIX =======================")
print(confusion_matrix(y_true, y_pred))

# Save trained model
os.makedirs('models', exist_ok=True)
model.save('models/mobilenetv2_animal_classifier.keras')
print("Model saved to models/mobilenetv2_animal_classifier.keras")